In [ ]:
import sys
sys.path.append("..")

In [ ]:
import numpy as np
import plotly.graph_objects as go
from scipy.signal.windows import gaussian

from wavelet_transform.postprocess import plotting as plot
import wavelet_transform.utils.training_data as train
from wavelet_transform.utils.machine_learning import train_k_means
import wavelet_transform.wpt.inverse_transform as inv

Input signal

In [ ]:
def plot_signal(data, time, title="Signal"):
    if not isinstance(data, list):
        data = [data]
    fig = go.Figure()
    for i in range(len(data)):
        d = data [i]
        fig.add_trace(
            go.Scatter(x=time, y=d)
        )
    fig.update_layout(
        xaxis=dict(title="Time [s]"),
        yaxis=dict(title="Amplitude [DL]"),
        title=title
    )
    fig.show()

In [ ]:
np.random.seed(10)

freq = 1e6
cycles = 50
sampling_rate = 200e6
window = False

t_end = cycles / freq
t = np.arange(0, t_end, 1/sampling_rate)
sine_wave = np.sin(2*np.pi*freq*t)
signal = train.add_noise(sine_wave, 30)
transient = gaussian(len(t), std=1)
signal += 2 * transient
signal_pp = train.preprocess(signal, window=window)

sine_wave_pp = train.preprocess(sine_wave, window=window)
plot_signal([signal_pp, sine_wave_pp], t)

Discrete wavelet transform

In [ ]:
# Parameters for the wavelet transform
wavelet = "db30"
levels = None  # set to None to use max-level transform
mode = "symmetric"
use_min_entropy = True
min_level = 0
freq_threshold = 1
no_clusters = (3, 6)

In [ ]:
wpt_df, transform = train.generate_training_data(
    signal_pp, levels, wavelet, mode, 
    include_tr_obj=True, min_entropy=use_min_entropy,
    min_level=min_level, freq_threshold=freq_threshold,
    keep_ratio="valid"
)

Training

In [ ]:
# k-means parameters
kmeans_params = {
    "init": "k-means++",
    "n_init":10,
    "max_iter":100,
    "tol":1e-8,
    "verbose":0,
    "algorithm":"lloyd",
}

# training data
classifying_metrics = [
    "log average energy",
    "kurtosis",
    # "crest factor",
    # "level entropy",
    "spectral entropy",
    "average energy",
    # "skewness"
]
# Set to no_clusters to None to select optimal number of clusters by silhouette score
scale_data = "standard"
wpt_df_train = wpt_df.copy()

In [ ]:
data_train, labels, data_train_normalised, centers, scaler, kmeans = train_k_means(
    wpt_df_train,
    classifying_metrics, 
    no_clusters=no_clusters, 
    kmeans_params=kmeans_params, 
    scale_data=scale_data
)

In [ ]:
ordered_centers, ordered_labels = plot.relabel(labels, centers, "average energy")

Plot results

In [ ]:
transform.plot_tree(node_colour=list(ordered_labels))

In [ ]:
fig = plot.plot_centers(classifying_metrics, ordered_centers)

In [ ]:
fig = plot.plot_clustering(
    data_train, 
    ordered_centers,
    ordered_labels, 
    classifying_metrics[:3], 
    title=None
)

In [ ]:
fig = plot.plot_reconstructions_by_group(transform, ordered_labels, t, freq_threshold)

In [ ]:
nodes = transform.get_leaf_nodes(freq_threshold=freq_threshold)

# Get highest kurtosis node
node_idx, kurt = nodes[0], 0
for n in nodes:
    nk = n.kurtosis
    if nk > kurt:
        node_idx, kurt = n, nk

rec = inv.inv_dwpt(transform, node_idx.level, nodes=node_idx)

In [ ]:
plot_signal(rec, t, f"High kurtosis signal: {node_idx.kurtosis:.2f}")